<a href="https://colab.research.google.com/github/feall/MinervaAI/blob/main/MinervaAI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install PyMuPDF
!pip install --upgrade PyMuPDF

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 31.6 MB/s eta 0:00:00


In [ ]:
import fitz
import re
import pandas as pd
import unicodedata

In [ ]:
INSCRICAO_PATTERN = re.compile(r'^\d{12}$') # Confere se tem numero de inscrição

def process_pdf(pdf_path: str) -> pd.DataFrame:
    doc = fitz.open(pdf_path)
    n_pages = len(doc) - 1 # Tira a ultima pagina de modalidade

    records = []
    for page_num in range(n_pages):
        lines = [
            l.strip()
            for l in doc[page_num].get_text("text").splitlines()
            if l.strip()
        ]

        i = 0
        while i < len(lines):
            if (i + 7) < len(lines) and INSCRICAO_PATTERN.match(lines[i + 5]):
                records.append({
                    "curso":      lines[i],
                    "formacao":   lines[i + 1],
                    "campus":     lines[i + 2],
                    "turno":      lines[i + 3],
                    #"nome":       lines[i + 4], Não preciso do nome da pessoa
                    "id":  lines[i + 5],
                    "nota":       float(lines[i + 6]),
                    "modalidade": int(lines[i + 7]),
                })
                i += 8 # Pula cabeçalho
            else:
                i += 1
    doc.close()

    return pd.DataFrame(records)

In [ ]:
df = process_pdf("/content/lista-de-espera.pdf")
#df.to_excel("lista_espera2.xlsx", index=False)
print(df)


                          curso                       formacao  \
0     ABI - CIÊNCIAS BIOLÓGICAS  Área Básica de Ingresso (ABI)   
1     ABI - CIÊNCIAS BIOLÓGICAS  Área Básica de Ingresso (ABI)   
2     ABI - CIÊNCIAS BIOLÓGICAS  Área Básica de Ingresso (ABI)   
3     ABI - CIÊNCIAS BIOLÓGICAS  Área Básica de Ingresso (ABI)   
4     ABI - CIÊNCIAS BIOLÓGICAS  Área Básica de Ingresso (ABI)   
...                         ...                            ...   
1138        TERAPIA OCUPACIONAL                    Bacharelado   
1139        TERAPIA OCUPACIONAL                    Bacharelado   
1140        TERAPIA OCUPACIONAL                    Bacharelado   
1141        TERAPIA OCUPACIONAL                    Bacharelado   
1142        TERAPIA OCUPACIONAL                    Bacharelado   

                    campus     turno            id    nota  modalidade  
0     Cidade Universitária  Integral  251036518826  602.38           1  
1     Cidade Universitária  Integral  241009234410  600.87   

Com o pymu foi bem facil extrair as notas, curso, turno e etc
So falta fazer isso em paralelo pra quando for fazer pra varias tabelas

Mas agora é melhor partir pra outros problemas por exemplo conseguir os pesos

In [ ]:
doc = fitz.open("/content/termo_adesao_2026.pdf")
page = doc[2]
text = page.get_text("text")

lines = text.splitlines()

for i, line in enumerate(lines):
    print(i, repr(line))

0 'MINISTÉRIO DA EDUCAÇÃO - MEC'
1 'Secretaria de Educação Superior - SESU'
2 'Sisu - Sistema de Seleção Unificada'
3 'Termo de Adesão 1ª edição de 2026'
4 'A autencidade deste documento pode ser conferida no site /visualizar-termo, informando os 7 (setes) primeiros dígitos da'
5 'autenticidade e o número de protocolo.'
6 'Autenticidade: 1543911FFA1A40D8C4E09216A3E81298E20139EC'
7 'Nº do protocolo: D18UX2M'
8 '3 / 192'
9 ' '
10 'Local de Oferta: 657978 - Cidade Universitária (Rio de Janeiro, RJ)'
11 'Avenida Brigadeiro Trompowsky, s/n - Ilha do Fundão - Rio de Janeiro - RJ, 21941-590 - (21) 3938-9900'
12 '14333 - ARQUITETURA E URBANISMO'
13 'Código: 14333'
14 ' Bacharelado'
15 'Grau:'
16 ' Integral (Matutino/Vespertino)'
17 'Turno:'
18 ' Semestral'
19 'Periodicidade:'
20 ' 10'
21 'Integralização:'
22 ' 240'
23 'Vagas autorizadas:'
24 ' 240 vagas, sendo 120 vagas no 1º'
25 'Vagas ofertadas no Sisu:'
26 'semestre e 120 vagas no 2º semestre.'
27 ' 50%'
28 'Percentual de vagas reservadas d

In [ ]:
def is_numeric(line):
    return line.replace(',', '').replace('.', '').strip().isdigit()

def process_pdf2(pdf_path: str) -> pd.DataFrame:
    doc = fitz.open(pdf_path)

    records = []
    for page_num in range(2,152):
        lines = [l.strip() for l in doc[page_num].get_text("text").splitlines() if l.strip()]

        for i, line in enumerate(lines):
            if line != 'Prova do Enem':
                continue

            # busca o nome do curso antes da linha 'Prova do Enem'
            nome_curso = None
            for k in range(i - 1, -1, -1):
                m = re.match(r'^\d+ - (.+)$', lines[k])
                if m:
                    nome_curso = m.group(1)
                    break

            if not nome_curso:
                continue

            # coleta os 5 pesos
            pesos = []
            j = i
            while len(pesos) < 5 and j < len(lines):
                if is_numeric(lines[j]):
                    pesos.append(float(lines[j].replace(',', '.')))
                    j += 2
                else:
                    j += 1

            if len(pesos) == 5:
                records.append({
                    "curso":    nome_curso,
                    "redacao":  pesos[0],
                    "nat":      pesos[1],
                    "hum":      pesos[2],
                    "ling":     pesos[3],
                    "mat":      pesos[4],
                })

    doc.close()
    return pd.DataFrame(records)

In [ ]:
df2 = process_pdf2("/content/termo_adesao_2026.pdf")
df2.drop_duplicates(inplace=True)
print(df2)
df2.to_json("cursos_pesos.json")


                            curso  redacao  nat  hum  ling  mat
0         ARQUITETURA E URBANISMO      3.0  1.0  4.0   2.0  4.0
1      ARTES CÊNICAS - CENOGRAFIA      3.0  1.0  2.0   2.0  1.0
2    ARTES CÊNICAS - INDUMENTÁRIA      3.0  1.0  2.0   2.0  1.0
3                   ARTES VISUAIS      3.0  1.0  2.0   2.0  1.0
4       ARTES VISUAIS - ESCULTURA      3.0  1.0  2.0   2.0  1.0
..                            ...      ...  ...  ...   ...  ...
142                    JORNALISMO      3.0  1.0  2.0   2.0  1.0
143                     PEDAGOGIA      3.0  1.0  1.0   1.0  1.0
146                    PSICOLOGIA      3.0  1.0  2.0   2.0  1.0
147       RELAÇÕES INTERNACIONAIS      3.0  1.0  2.0   3.0  1.0
148                SERVIÇO SOCIAL      3.0  1.0  2.0   1.0  1.0

[110 rows x 6 columns]


agora o proximo passo é a partir da nota da pessoa calcular a media dela pra cada curso

é um produto interno dos vetores notas x pesos da pra fazer com o numpy

In [ ]:
import numpy as np

notas = [ 700,200,600,600,600] #red, nat, hum, ling, mat
pesos = df2[["redacao","nat","hum","ling","mat"]].to_numpy()

print(np.sum(notas*pesos[0])/np.sum(pesos[0]))


592.8571428571429


Tava trabalhando no VScode e tive avanços na parte de pegar os cursos e pesos mas ainda to tendo trabalho na tabela resolvi voltar pro colab e vamos de codigo novo vibe codado

In [ ]:
# --- Referências ---
cursos_ref = pd.read_json("cursos_pesos.json")["curso"].tolist()
campus_ref = [
    "Cidade Universitária",
    "Campus UFRJ-Duque de Caxias Prof. Geraldo Cidade",
    "Centro Multidisciplinar de Macaé",
    "Faculdade de Direito",
    "Instituto de Biodiversidade e Sustentabilidade - NUPEM",
    "Instituto de Psiquiatria da UFRJ - IPUB",
    "LARGO SÃO FRANCISCO",
    "Observatório do Valongo",
    "Praia Vermelha"
]
turno_ref = ["Integral", "Matutino", "Vespertino", "Noturno"]
grau_ref = ["Bacharelado", "Licenciatura", "Área Básica de Ingresso (ABI)"]

def norm(s: str) -> str:
    return "".join(c for c in unicodedata.normalize("NFD", s.strip().lower())
                   if unicodedata.category(c) != "Mn")

def process_pdf(pdf_path: str, start: int = 0, end: int = None) -> pd.DataFrame:
    doc = fitz.open(pdf_path)
    if end is None:
        end = len(doc)

    records = []

    # 🔹 Loop pelos cursos do JSON primeiro para manter a ordem
    for curso in cursos_ref:
        curso_norm = norm(curso)
        found = False
        for page_num in range(start, end):
            lines = [l.strip() for l in doc[page_num].get_text("text").splitlines() if l.strip()]
            # Match bidirecional
            if any(curso_norm in norm(line) or norm(line) in curso_norm for line in lines):
                records.append({"nome": curso})
                found = True
                break  # passa para o próximo curso do JSON
    doc.close()
    return pd.DataFrame(records)


pdf_path = "lista-de-espera.pdf"

df = process_pdf(pdf_path, 0, 1)

#df.to_json("lista_espera.json", orient="records", indent=4, force_ascii=False)

print(df)

                                                nome
0                          ABI - CIÊNCIAS BIOLÓGICAS
1   BIBLIOTECONOMIA E GESTÃO DE UNIDADES DE INFOR...
2                                CIÊNCIAS BIOLÓGICAS
3                                CIÊNCIAS BIOLÓGICAS
4             CIÊNCIAS BIOLÓGICAS: MODALIDADE MÉDICA
5              ENGENHARIA DE COMPUTAÇÃO E INFORMAÇÃO
6                                CIÊNCIAS BIOLÓGICAS
7                                CIÊNCIAS BIOLÓGICAS
8                                      ADMINISTRAÇÃO
9   BIBLIOTECONOMIA E GESTÃO DE UNIDADES DE INFOR...


In [ ]:
import fitz
import pandas as pd
import unicodedata

norm = lambda s: ''.join(c for c in unicodedata.normalize("NFD", str(s).strip().lower())
                        if unicodedata.category(c) != 'Mn')

cursos_ref = pd.read_json("cursos_pesos.json")["curso"].tolist()

def process_pdf_first_page_preciso(pdf_path):
    doc = fitz.open(pdf_path)
    page = doc[0]  # primeira página
    lines = [l.strip() for l in page.get_text("text").splitlines() if l.strip()]
    records = []

    for line in lines:
        line_norm = norm(line)
        # Ignora linhas muito curtas (menos de 5 caracteres)
        if len(line_norm) < 5:
            continue

        for curso in cursos_ref:
            curso_norm = norm(curso)
            # Match apenas se todo o nome do curso estiver contido na linha
            if curso_norm in line_norm:
                records.append({"nome": curso})
                break  # para no primeiro match da linha

    doc.close()
    return pd.DataFrame(records)

df = process_pdf_first_page_preciso("lista-de-espera.pdf")
print(df)

                          nome
0    ABI - CIÊNCIAS BIOLÓGICAS
1    ABI - CIÊNCIAS BIOLÓGICAS
2    ABI - CIÊNCIAS BIOLÓGICAS
3    ABI - CIÊNCIAS BIOLÓGICAS
4    ABI - CIÊNCIAS BIOLÓGICAS
5    ABI - CIÊNCIAS BIOLÓGICAS
6    ABI - CIÊNCIAS BIOLÓGICAS
7    ABI - CIÊNCIAS BIOLÓGICAS
8    ABI - CIÊNCIAS BIOLÓGICAS
9    ABI - CIÊNCIAS BIOLÓGICAS
10   ABI - CIÊNCIAS BIOLÓGICAS
11   ABI - CIÊNCIAS BIOLÓGICAS
12   ABI - CIÊNCIAS BIOLÓGICAS
13               ADMINISTRAÇÃO
14               ADMINISTRAÇÃO
15               ADMINISTRAÇÃO
16               ADMINISTRAÇÃO
17               ADMINISTRAÇÃO
18               ADMINISTRAÇÃO
19               ADMINISTRAÇÃO
20               ADMINISTRAÇÃO
21               ADMINISTRAÇÃO


In [ ]:

# --- Normalização ---
def norm(s: str) -> str:
    """Remove acentos, espaços extras e coloca em minúsculas"""
    return "".join(c for c in unicodedata.normalize("NFD", str(s).strip().lower())
                   if unicodedata.category(c) != "Mn")

# --- Carrega referências ---
cursos_ref = pd.read_json("cursos_pesos.json")["curso"].tolist()
campus_ref = [
    "Cidade Universitária",
    "Campus UFRJ-Duque de Caxias Prof. Geraldo Cidade",
    "Centro Multidisciplinar de Macaé",
    "Faculdade de Direito",
    "Instituto de Biodiversidade e Sustentabilidade - NUPEM",
    "Instituto de Psiquiatria da UFRJ - IPUB",
    "LARGO SÃO FRANCISCO",
    "Observatório do Valongo",
    "Praia Vermelha"
]
turno_ref = ["Integral", "Matutino", "Vespertino", "Noturno"]
grau_ref = ["Bacharelado", "Licenciatura", "Área Básica de Ingresso (ABI)"]

# --- Função principal ---
def process_pdf_final(pdf_path: str, start: int = 0, end: int = None) -> pd.DataFrame:
    """
    Processa o PDF de lista de espera e retorna DataFrame com coluna 'nome' de cursos encontrados.
    - Match seguro com JSON
    - Preserva ordem do PDF
    - Ignora linhas curtas (prováveis cabeçalhos/rodapés)
    """
    doc = fitz.open(pdf_path)
    if end is None:
        end = len(doc)

    records = []

    for page_num in range(start, end):
        lines = [l.strip() for l in doc[page_num].get_text("text").splitlines() if l.strip()]

        for line in lines:
            line_norm = norm(line)
            if len(line_norm) < 5:  # ignora linhas curtas
                continue

            for curso in cursos_ref:
                curso_norm = norm(curso)
                if curso_norm in line_norm:  # match seguro
                    records.append({"nome": curso})
                    break  # pára no primeiro match da linha

    doc.close()
    return pd.DataFrame(records)

# --- Exemplo de uso no Colab ---
# Suba o PDF para o Colab e indique o caminho
pdf_path = "lista-de-espera.pdf"
df = process_pdf_final(pdf_path, start=0, end=51)

# Salva o resultado
#df.to_json("lista_espera_final.json", orient="records", indent=4, force_ascii=False)
df.to_excel("lista_espera_final.xlsx", index=False)

In [ ]:

# --- Normalização ---
def norm(s: str) -> str:
    """Remove acentos, espaços extras e coloca em minúsculas"""
    return "".join(c for c in unicodedata.normalize("NFD", str(s).strip().lower())
                   if unicodedata.category(c) != "Mn")

# --- Carrega cursos ---
cursos_ref = pd.read_json("cursos_pesos.json")["curso"].tolist()

# --- Função para extrair o curso de uma linha ---
def match_curso(line: str, cursos: list) -> str:
    """
    Recebe uma linha do PDF e retorna o curso correspondente do JSON
    - Prioriza nomes maiores para evitar sobreposição
    """
    line_norm = norm(line)
    for curso in sorted(cursos, key=lambda x: -len(x)):  # maior nome primeiro
        if norm(curso) in line_norm:
            return curso
    return None

# --- Função principal simplificada ---
def process_pdf_nome(pdf_path: str, start: int = 0, end: int = None) -> pd.DataFrame:
    doc = fitz.open(pdf_path)
    if end is None:
        end = len(doc)

    records = []

    for page_num in range(start, end):
        lines = [l.strip() for l in doc[page_num].get_text("text").splitlines() if l.strip()]
        for line in lines:
            # ignora linhas muito curtas (prováveis cabeçalhos/rodapés)
            if len(norm(line)) < 5:
                continue

            curso = match_curso(line, cursos_ref)
            if curso:
                records.append({"nome": curso})

    doc.close()
    return pd.DataFrame(records)

# --- Exemplo de uso ---
pdf_path = "lista-de-espera.pdf"  # Suba o PDF no Colab
df = process_pdf_nome(pdf_path, start=0, end=52)

# Salva resultado
df.to_excel("lista_espera.xlsx", index=False)

Essa proxima versao funcionou perfeitamente 😇 nao sei o porque depois tenho que ir testando

In [ ]:
# --- Normalização ---
def norm(s: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFD", str(s).strip().lower())
        if unicodedata.category(c) != "Mn"
    )

# --- Similaridade simples ---
def match_score(line_norm, curso_norm):
    """
    Mede quanto do curso aparece na linha
    """
    if curso_norm in line_norm:
        return len(curso_norm) / len(line_norm)
    if line_norm in curso_norm:
        return len(line_norm) / len(curso_norm)
    return 0

# --- Match seguro REAL ---
def match_curso_strict(line: str, cursos: list) -> str:
    line_norm = norm(line)

    best_match = None
    best_score = 0

    for curso in cursos:
        curso_norm = norm(curso)
        score = match_score(line_norm, curso_norm)

        # 🔥 threshold importante
        if score > 0.6 and score > best_score:
            best_score = score
            best_match = curso

    return best_match

# --- Função principal ---
def process_pdf(pdf_path: str, start: int = 0, end: int = None) -> pd.DataFrame:
    doc = fitz.open(pdf_path)
    if end is None:
        end = len(doc)

    records = []

    for page_num in range(start, end):
        lines = [l.strip() for l in doc[page_num].get_text("text").splitlines() if l.strip()]

        i = 0
        while i < len(lines):
            line = lines[i]
            line_norm = norm(line)

            if len(line_norm) < 5:
                i += 1
                continue

            # tenta linha normal
            curso = match_curso_strict(line, cursos_ref)

            # tenta linha + próxima (curso quebrado)
            if not curso and i + 1 < len(lines):
                combined = line + " " + lines[i + 1]
                curso = match_curso_strict(combined, cursos_ref)

                if curso:
                    records.append({"nome": curso})
                    i += 2
                    continue

            if curso:
                records.append({"nome": curso})

            i += 1

    doc.close()
    return pd.DataFrame(records)


# --- carregar cursos ---
cursos_ref = pd.read_json("cursos_pesos.json")["curso"].tolist()

# --- rodar ---
df = process_pdf("lista-de-espera.pdf", 0, 52)

# Salva resultado
df.to_excel("lista_espera4.xlsx", index=False)
print("✅ ORDEM CORRIGIDA!")


✅ ORDEM CORRIGIDA!


agora o proximo passo é adicionar os outros atributos: modalidade, turno, grau, nota e campus

vamos por campus que parece ser mais complexo

In [ ]:

# --- Normalização ---
def norm(s: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFD", str(s).strip().lower())
        if unicodedata.category(c) != "Mn"
    )

# --- Referências ---
cursos_ref = pd.read_json("cursos_pesos.json")["curso"].tolist()

campus_ref = [
    "Cidade Universitária",
    "Campus UFRJ-Duque de Caxias Prof. Geraldo Cidade",
    "Centro Multidisciplinar de Macaé",
    "Faculdade de Direito",
    "Instituto de Biodiversidade e Sustentabilidade - NUPEM",
    "Instituto de Psiquiatria da UFRJ - IPUB",
    "LARGO SÃO FRANCISCO",
    "Observatório do Valongo",
    "Praia Vermelha"
]

# --- Match curso (o seu já corrigido) ---
def match_curso_strict(line: str, cursos: list) -> str:
    line_norm = norm(line)

    best_match = None
    best_score = 0

    for curso in cursos:
        curso_norm = norm(curso)

        if curso_norm in line_norm:
            score = len(curso_norm) / len(line_norm)
        elif line_norm in curso_norm:
            score = len(line_norm) / len(curso_norm)
        else:
            score = 0

        if score > 0.6 and score > best_score:
            best_score = score
            best_match = curso

    return best_match

# --- Match campus ---
def match_campus(line: str, campus_list: list) -> str:
    line_norm = norm(line)
    for campus in campus_list:
        if norm(campus) in line_norm:
            return campus
    return None

# --- Parser ---
def process_pdf(pdf_path: str, start=0, end=None):
    doc = fitz.open(pdf_path)
    if end is None:
        end = len(doc)

    records = []
    current = None

    for page_num in range(start, end):
        lines = [l.strip() for l in doc[page_num].get_text("text").splitlines() if l.strip()]

        i = 0
        while i < len(lines):
            line = lines[i]
            line_norm = norm(line)

            if len(line_norm) < 5:
                i += 1
                continue

            # --- detectar curso ---
            curso = match_curso_strict(line, cursos_ref)

            if not curso and i + 1 < len(lines):
                combined = line + " " + lines[i + 1]
                curso = match_curso_strict(combined, cursos_ref)

                if curso:
                    # salva anterior
                    if current:
                        records.append(current)

                    current = {"nome": curso, "campus": None}
                    i += 2
                    continue

            if curso:
                if current:
                    records.append(current)

                current = {"nome": curso, "campus": None}
                i += 1
                continue

            # --- detectar campus ---
            if current and current["campus"] is None:
                campus = match_campus(line, campus_ref)
                if campus:
                    current["campus"] = campus

            i += 1

    # adiciona último
    if current:
        records.append(current)

    doc.close()
    return pd.DataFrame(records)


# --- EXECUÇÃO ---
df = process_pdf("lista-de-espera.pdf", 0, 52)

# Salva resultado
df.to_excel("lista_espera5.xlsx", index=False)
print("✅ LESGOOOO")

✅ LESGOOOO


In [ ]:
import fitz  # PyMuPDF
import pandas as pd
import unicodedata

# --- Normalização ---
def norm(s: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFD", str(s).strip().lower())
        if unicodedata.category(c) != "Mn"
    )

# --- Match curso ---
def match_curso_strict(line: str, cursos: list) -> str:
    line_norm = norm(line)
    best_match = None
    best_score = 0
    for curso in cursos:
        curso_norm = norm(curso)
        if curso_norm in line_norm:
            score = len(curso_norm) / len(line_norm)
        elif line_norm in curso_norm:
            score = len(line_norm) / len(curso_norm)
        else:
            score = 0
        if score > 0.6 and score > best_score:
            best_score = score
            best_match = curso  # retorna o nome completo do JSON
    return best_match

# --- Match campus com substrings únicas ---
def match_campus(line: str) -> str:
    line_norm = norm(line)
    # dicionário de substrings únicas -> nome completo
    substrings = {
        "Duque de Caxias": "Campus UFRJ-Duque de Caxias Prof. Geraldo Cidade",
        "Biodiversidade": "Instituto de Biodiversidade e Sustentabilidade - NUPEM",
        "Macaé": "Centro Multidisciplinar de Macaé",
        "Cidade Universitária": "Cidade Universitária",
        "Faculdade de Direito": "Faculdade de Direito",
        "IPUB": "Instituto de Psiquiatria da UFRJ - IPUB",
        "LARGO SÃO FRANCISCO": "LARGO SÃO FRANCISCO",
        "Valongo": "Observatório do Valongo",
        "Praia Vermelha": "Praia Vermelha"
    }
    for sub, full in substrings.items():
        if norm(sub) in line_norm:
            return full
    return None

# --- Parser principal ---
def process_pdf(pdf_path: str, cursos_ref: list, start=0, end=None):
    doc = fitz.open(pdf_path)
    if end is None:
        end = len(doc)

    records = []
    current = None

    for page_num in range(start, end):
        lines = [l.strip() for l in doc[page_num].get_text("text").splitlines() if l.strip()]

        i = 0
        while i < len(lines):
            line = lines[i]

            # --- detectar curso ---
            curso = match_curso_strict(line, cursos_ref)

            # tenta combinar com a próxima linha caso o curso seja longo
            if not curso and i + 1 < len(lines):
                combined = line + " " + lines[i + 1]
                curso = match_curso_strict(combined, cursos_ref)
                if curso:
                    if current:
                        records.append(current)
                    current = {"nome": curso, "campus": None}
                    i += 2
                    continue

            if curso:
                if current:
                    records.append(current)
                current = {"nome": curso, "campus": None}
                i += 1
                continue

            # --- detectar campus ---
            if current and current["campus"] is None:
                campus = match_campus(line)
                if campus:
                    current["campus"] = campus

            i += 1

    if current:
        records.append(current)

    doc.close()
    return pd.DataFrame(records)

# --- CARREGAR REFERÊNCIAS ---
cursos_ref = pd.read_json("cursos_pesos.json")["curso"].tolist()

# --- EXECUTAR ---
df = process_pdf("lista-de-espera.pdf", cursos_ref, start=0, end=52)

# --- SALVAR RESULTADO ---
df.to_excel("lista_espera10.xlsx", index=False)
print("✅ NOME + CAMPUS EXTRAÍDOS COM SUCESSO!")


✅ NOME + CAMPUS EXTRAÍDOS COM SUCESSO!


In [ ]:
agora foi o campus e o nome
agora vou tentar o turno

In [ ]:
import fitz  # PyMuPDF
import pandas as pd
import unicodedata

# --- Normalização ---
def norm(s: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFD", str(s).strip().lower())
        if unicodedata.category(c) != "Mn"
    )

# --- Match curso ---
def match_curso_strict(line: str, cursos: list) -> str:
    line_norm = norm(line)
    best_match = None
    best_score = 0
    for curso in cursos:
        curso_norm = norm(curso)
        if curso_norm in line_norm:
            score = len(curso_norm) / len(line_norm)
        elif line_norm in curso_norm:
            score = len(line_norm) / len(curso_norm)
        else:
            score = 0
        if score > 0.6 and score > best_score:
            best_score = score
            best_match = curso  # retorna o nome completo do JSON
    return best_match

# --- Match campus com substrings únicas ---
def match_campus(line: str) -> str:
    line_norm = norm(line)
    substrings = {
        "Duque de Caxias": "Campus UFRJ-Duque de Caxias Prof. Geraldo Cidade",
        "Biodiversidade": "Instituto de Biodiversidade e Sustentabilidade - NUPEM",
        "Macaé": "Centro Multidisciplinar de Macaé",
        "Cidade Universitária": "Cidade Universitária",
        "Faculdade de Direito": "Faculdade de Direito",
        "IPUB": "Instituto de Psiquiatria da UFRJ - IPUB",
        "LARGO SÃO FRANCISCO": "LARGO SÃO FRANCISCO",
        "Valongo": "Observatório do Valongo",
        "Praia Vermelha": "Praia Vermelha"
    }
    for sub, full in substrings.items():
        if norm(sub) in line_norm:
            return full
    return None

# --- Match turno ---
def match_turno(line: str) -> str:
    line_norm = norm(line)
    turno_list = ["integral", "matutino", "vespertino", "noturno"]
    for t in turno_list:
        if t in line_norm:
            return t.capitalize()  # Mantém com letra maiúscula inicial
    return None

# --- Parser principal ---
def process_pdf(pdf_path: str, cursos_ref: list, start=0, end=None):
    doc = fitz.open(pdf_path)
    if end is None:
        end = len(doc)

    records = []
    current = None

    for page_num in range(start, end):
        lines = [l.strip() for l in doc[page_num].get_text("text").splitlines() if l.strip()]

        i = 0
        while i < len(lines):
            line = lines[i]

            # --- detectar curso ---
            curso = match_curso_strict(line, cursos_ref)

            # tenta combinar com a próxima linha caso o curso seja longo
            if not curso and i + 1 < len(lines):
                combined = line + " " + lines[i + 1]
                curso = match_curso_strict(combined, cursos_ref)
                if curso:
                    if current:
                        records.append(current)
                    current = {"nome": curso, "campus": None, "turno": None}
                    i += 2
                    continue

            if curso:
                if current:
                    records.append(current)
                current = {"nome": curso, "campus": None, "turno": None}
                i += 1
                continue

            # --- detectar campus ---
            if current and current["campus"] is None:
                campus = match_campus(line)
                if campus:
                    current["campus"] = campus

            # --- detectar turno ---
            if current and current["turno"] is None:
                turno = match_turno(line)
                if turno:
                    current["turno"] = turno

            i += 1

    if current:
        records.append(current)

    doc.close()
    return pd.DataFrame(records)

# --- CARREGAR REFERÊNCIAS ---
cursos_ref = pd.read_json("cursos_pesos.json")["curso"].tolist()

# --- EXECUTAR ---
df = process_pdf("lista-de-espera.pdf", cursos_ref, start=0, end=52)

# --- SALVAR RESULTADO ---
df.to_excel("lista_espera11.xlsx", index=False)
print("✅ NOME + CAMPUS + TURNO EXTRAÍDOS COM SUCESSO!")

deu certo novamente !
agora falta modalidade, grau, nota

vamos partir pra grau

In [ ]:

# --- Normalização ---
def norm(s: str) -> str:
    return "".join(
        c for c in unicodedata.normalize("NFD", str(s).strip().lower())
        if unicodedata.category(c) != "Mn"
    )

# --- Match curso ---
def match_curso_strict(line: str, cursos: list) -> str:
    line_norm = norm(line)
    best_match = None
    best_score = 0
    for curso in cursos:
        curso_norm = norm(curso)
        if curso_norm in line_norm:
            score = len(curso_norm) / len(line_norm)
        elif line_norm in curso_norm:
            score = len(line_norm) / len(curso_norm)
        else:
            score = 0
        if score > 0.6 and score > best_score:
            best_score = score
            best_match = curso  # retorna o nome completo do JSON
    return best_match

# --- Match campus ---
def match_campus(line: str) -> str:
    line_norm = norm(line)
    substrings = {
        "Duque de Caxias": "Campus UFRJ-Duque de Caxias Prof. Geraldo Cidade",
        "Biodiversidade": "Instituto de Biodiversidade e Sustentabilidade - NUPEM",
        "Macaé": "Centro Multidisciplinar de Macaé",
        "Cidade Universitária": "Cidade Universitária",
        "Faculdade de Direito": "Faculdade de Direito",
        "IPUB": "Instituto de Psiquiatria da UFRJ - IPUB",
        "LARGO SÃO FRANCISCO": "LARGO SÃO FRANCISCO",
        "Valongo": "Observatório do Valongo",
        "Praia Vermelha": "Praia Vermelha"
    }
    for sub, full in substrings.items():
        if norm(sub) in line_norm:
            return full
    return None

# --- Match turno ---
def match_turno(line: str) -> str:
    line_norm = norm(line)
    turno_list = ["integral", "matutino", "vespertino", "noturno"]
    for t in turno_list:
        if t in line_norm:
            return t.capitalize()
    return None

# --- Match grau ---
def match_grau(line: str) -> str:
    line_norm = norm(line)
    grau_list = {
        "bacharelado": "Bacharelado",
        "licenciatura": "Licenciatura",
        "abi": "Área Básica de Ingresso (ABI)"
    }
    for key, value in grau_list.items():
        if key in line_norm:
            return value
    return None

# --- Parser principal ---
def process_pdf(pdf_path: str, cursos_ref: list, start=0, end=None):
    doc = fitz.open(pdf_path)
    if end is None:
        end = len(doc)

    records = []
    current = None

    for page_num in range(start, end):
        lines = [l.strip() for l in doc[page_num].get_text("text").splitlines() if l.strip()]

        i = 0
        while i < len(lines):
            line = lines[i]

            # --- detectar curso ---
            curso = match_curso_strict(line, cursos_ref)

            # tenta combinar com a próxima linha caso o curso seja longo
            if not curso and i + 1 < len(lines):
                combined = line + " " + lines[i + 1]
                curso = match_curso_strict(combined, cursos_ref)
                if curso:
                    if current:
                        records.append(current)
                    current = {"nome": curso, "campus": None, "turno": None, "grau": None}
                    i += 2
                    continue

            if curso:
                if current:
                    records.append(current)
                current = {"nome": curso, "campus": None, "turno": None, "grau": None}
                i += 1
                continue

            # --- detectar campus ---
            if current and current["campus"] is None:
                campus = match_campus(line)
                if campus:
                    current["campus"] = campus

            # --- detectar turno ---
            if current and current["turno"] is None:
                turno = match_turno(line)
                if turno:
                    current["turno"] = turno

            # --- detectar grau ---
            if current and current["grau"] is None:
                grau = match_grau(line)
                if grau:
                    current["grau"] = grau

            i += 1

    if current:
        records.append(current)

    doc.close()
    return pd.DataFrame(records)

# --- CARREGAR REFERÊNCIAS ---
cursos_ref = pd.read_json("cursos_pesos.json")["curso"].tolist()

# --- EXECUTAR ---
df = process_pdf("lista-de-espera.pdf", cursos_ref, start=0, end=52)

# --- SALVAR RESULTADO ---
df.to_excel("lista_espera.xlsx", index=False)
print("✅ NOME + CAMPUS + TURNO + GRAU EXTRAÍDOS COM SUCESSO!")


✅ NOME + CAMPUS + TURNO + GRAU EXTRAÍDOS COM SUCESSO!


agora adicionar nota

tentei mas deu errado

acho que da pra fazer sem o vibe code mas deixar pra amanha

era bom pegar o codigo anterior e pedir pra simplificar ao maximo antes de aclopar com o de pegar a nota

da pra fazer da modalidade separado tambem é sempre uma linha unica com um unico algorismo


NEW DAY!!!!
Pipeline pra da certo:
1- Criar um script pra extrair somente as notas
2 - Adicionar nesse script a modalidade
3 - Adicionar a origem ou seja de qual pdf veio ANO.PERIODO.CHAMADA
-------------------------------------------------------------
4 - Pegar o script que pega o NOME CAMPUS TURNO e GRAU e deixar ele mais simples possivel mas que ainda de certo
5 - Fazer o mesmo pro script novo
6 - Juntar eles e depois simplificar novamente
--------------------------------------------------------------
7 - Testar pra outras 2 listas procurar por paginas diferentes por exemplo a ultima pagina com conteudo util pode conter tambem conteudo nao util e isso poderia quebrar o scrapping quando fosse processar varios pdf

8 - Crawling das listas de espera

9 - Pegar so do ultimo ano pra testar ate aqui teria funcionado tudo pra qualquer pdf entao esse passo seria mais unir o crawling com scrapping e testar pra poder escalar depois
-------------------------------------------------------------------------

Se tudo der certo é melhor parar por aqui


In [ ]:

# --- Regex da nota ---
REGEX_NOTA = re.compile(r"\b\d{3}\.\d{2}\b")

# --- Detecta se é modalidade (número pequeno inteiro) ---
def is_modalidade(line: str) -> bool:
    return line.isdigit() and 0 <= int(line) <= 10


# --- Função principal ---
def extract_notas_modalidade(pdf_path: str, start: int = 0, end: int = None) -> pd.DataFrame:
    doc = fitz.open(pdf_path)
    if end is None:
        end = len(doc)

    records = []

    for page_num in range(start, end):
        lines = [l.strip() for l in doc[page_num].get_text("text").splitlines() if l.strip()]

        i = 0
        while i < len(lines):
            line = lines[i]

            # --- PROCURA NOTA ---
            match = REGEX_NOTA.search(line)

            if match:
                nota = float(match.group())

                modalidade = None

                # --- tenta pegar modalidade na próxima linha ---
                if i + 1 < len(lines):
                    next_line = lines[i + 1].strip()

                    if is_modalidade(next_line):
                        modalidade = int(next_line)
                        i += 1  # pula linha da modalidade

                # --- fallback (caso modalidade esteja na mesma linha) ---
                if modalidade is None:
                    parts = line.split()
                    for p in parts:
                        if p.isdigit() and is_modalidade(p):
                            modalidade = int(p)
                            break

                # --- salva registro ---
                records.append({
                    "nota": nota,
                    "modalidade": modalidade,
                })


            i += 1

    doc.close()
    return pd.DataFrame(records)


# --- USO ---
df_notas = extract_notas_modalidade(
    "lista-de-espera.pdf",
    start=0,
    end=52,
)

# salvar
df_notas.to_excel("notas_modalidade2.xlsx", index = False)

print("✅ EXTRAÇÃO DE NOTAS FINALIZADA!")

✅ EXTRAÇÃO DE NOTAS FINALIZADA!


deu quase certo mas faltou linha no final

In [ ]:
# Regex para notas no formato NNN.NN
REGEX_NOTA = re.compile(r"\b\d{3}\.\d{2}\b")

# Função para checar se é modalidade
def is_modalidade(line: str) -> bool:
    return line.isdigit() and 0 <= int(line) <= 10

# Função principal de extração de notas + modalidade
def extract_notas_modalidade(pdf_path: str, start: int = 0, end: int = 53) -> pd.DataFrame:
    doc = fitz.open(pdf_path)
    records = []

    for page_num in range(start, end):
        # pega todas as linhas não vazias da página
        lines = [l.strip() for l in doc[page_num].get_text("text").splitlines() if l.strip()]

        i = 0
        while i < len(lines):
            line = lines[i]
            nota = None
            modalidade = None

            match = REGEX_NOTA.search(line)
            if match:
                nota = float(match.group())

                # tenta pegar modalidade na próxima linha
                if i + 1 < len(lines):
                    next_line = lines[i + 1].strip()
                    if is_modalidade(next_line):
                        modalidade = int(next_line)
                        i += 1  # pula linha da modalidade

                # fallback: modalidade na mesma linha
                if modalidade is None:
                    for p in line.split():
                        if is_modalidade(p):
                            modalidade = int(p)
                            break

                # salva registro
                records.append({"nota": nota, "modalidade": modalidade})

            i += 1

    doc.close()
    return pd.DataFrame(records)

# --- USO ---
pdf_path = "lista-de-espera.pdf"
df_notas = extract_notas_modalidade(pdf_path)

# Salva direto em Excel
df_notas.to_excel("notas_modalidade3.xlsx", index=False)

print("✅ Extração completa! Arquivo 'notas_modalidade3.xlsx' criado.")

✅ Extração completa! Arquivo 'notas_modalidade3.xlsx' criado.


vai ser melhor usar ancora ou combinar ancora com regex e etc

In [ ]:

def inspect_pdf_lines(pdf_path: str, start: int = 0, end: int = None, max_lines_per_page: int = 69):
    """
    Função de inspeção para visualizar linhas do PDF com índice.
    Útil para achar âncoras e padrões.

    pdf_path: caminho do PDF
    start, end: páginas para inspecionar
    max_lines_per_page: limita a visualização por página
    """
    doc = fitz.open(pdf_path)
    if end is None:
        end = len(doc)

    for page_num in range(start, end):
        print(f"\n--- Página {page_num} ---")
        lines = [l.strip() for l in doc[page_num].get_text("text").splitlines() if l.strip()]

        for idx, line in enumerate(lines[:max_lines_per_page]):
            print(f"{idx:03d}: {line}")

    doc.close()

# --- USO ---
pdf_path = "lista-de-espera.pdf"
inspect_pdf_lines(pdf_path, start=0, end=52)  # exibe as 5 primeiras páginas


--- Página 0 ---
000: CURSO
001: FORMAÇÃO
002: CAMPUS
003: TURNO
004: NOME
005: INSCRIÇÃO ENEM
006: NOTA
007: MODALIDADE
008: CLASSIFICAÇÃO
009: (ver legenda)
010: ABI - CIÊNCIAS BIOLÓGICAS
011: Área Básica de Ingresso (ABI)
012: Cidade Universitária
013: Integral
014: MARIANNE ROCHELLY DOS SANTOS SILVA
015: 251036518826
016: 602.38
017: 1
018: ABI - CIÊNCIAS BIOLÓGICAS
019: Área Básica de Ingresso (ABI)
020: Cidade Universitária
021: Integral
022: YASMIN APARECIDA LEUTERIO DUARTE
023: 241009234410
024: 600.87
025: 1
026: ABI - CIÊNCIAS BIOLÓGICAS
027: Área Básica de Ingresso (ABI)
028: Cidade Universitária
029: Integral
030: NICOLAS BRYNER ARAUJO DE PAULO
031: 231007650089
032: 600.8
033: 1
034: ABI - CIÊNCIAS BIOLÓGICAS
035: Área Básica de Ingresso (ABI)
036: Cidade Universitária
037: Integral
038: ANA CAROLINA EWERTON COUTINHO
039: 251001793248
040: 631.13
041: 5
042: ABI - CIÊNCIAS BIOLÓGICAS
043: Área Básica de Ingresso (ABI)
044: Cidade Universitária
045: Integral
046: VITOR HUG

In [ ]:


def extract_notas_modalidade_robusto(pdf_path, start=0, end=None):
    """
    Extrai nota e modalidade do PDF de forma robusta.
    Detecta cabeçalhos por página, varre todos os blocos de 8 linhas
    e preserva a ordem completa do PDF.
    """
    doc = fitz.open(pdf_path)
    if end is None:
        end = len(doc)

    records = []

    for page_num in range(start, end):
        # pega todas as linhas não vazias
        lines = [l.strip() for l in doc[page_num].get_text("text").splitlines() if l.strip()]

        # detecta cabeçalho
        if page_num == 0:
            # primeira página termina em '(ver legenda)'
            try:
                header_idx = next(i for i, l in enumerate(lines) if '(ver legenda)' in l) + 1
            except StopIteration:
                header_idx = 10  # fallback
        else:
            # outras páginas terminam em 'CONCURSO SISU/MEC 2026'
            try:
                header_idx = next(i for i, l in enumerate(lines) if 'CONCURSO SISU/MEC 2026' in l) + 1
            except StopIteration:
                header_idx = 13  # fallback

        content_lines = lines[header_idx:]

        # varre blocos de 8 linhas até o final da página
        i = 0
        while i + 7 < len(content_lines):  # precisa de pelo menos 8 linhas
            bloco = content_lines[i:i+8]
            # linha 6 = nota, linha 7 = modalidade
            try:
                nota = float(bloco[6]) if bloco[6] else None
            except:
                nota = None
            try:
                modalidade = int(bloco[7]) if bloco[7] else None
            except:
                modalidade = None

            records.append({
                "nota": nota,
                "modalidade": modalidade
            })
            i += 8  # próximo bloco

    doc.close()
    return pd.DataFrame(records)

# --- Uso no Colab ---
pdf_path = "/content/lista-de-espera.pdf"  # coloque o caminho do PDF
df_notas = extract_notas_modalidade_robusto(pdf_path, start=0, end=52)


# salva em Excel
df_notas.to_excel("/content/notas_modalidade5.xlsx", index=False)

deu tudo errado vamos mudar o approach denovo
no fitz da pra pegar informaçoes via boxes que seria o ideal na verdade pro tipo de documento que é aquela tabela